# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
#loading the data
!pip install -q duckdb huggingface_hub

import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
base = "hf://datasets/FlyRank/internship-warehouse"

def load_month_agg(month_str):
    return con.sql(f"""
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS gsc_impressions,
               SUM(gsc_clicks) AS gsc_clicks,
               SUM(gsc_sum_position) AS gsc_sum_position,
               SUM(ga4_sessions) AS ga4_sessions,
               SUM(ga4_engaged_sessions) AS ga4_engaged_sessions
        FROM read_parquet('{base}/fact_content_daily_performance/month={month_str}/data_0.parquet')
        GROUP BY client_hash_id, content_hash_id
    """).df()

df_feb_agg = load_month_agg('2026-02')
df_march_agg = load_month_agg('2026-03')

df_april_agg = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS gsc_clicks_apr
    FROM read_parquet('{base}/fact_content_daily_performance/month=2026-04/data_0.parquet')
    GROUP BY client_hash_id, content_hash_id
""").df()

print(f"Feb: {len(df_feb_agg)}, March: {len(df_march_agg)}, April(clicks only): {len(df_april_agg)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feb: 321546, March: 331437, April(clicks only): 362172


In [2]:
# rule code from previous weeks
def position_bucket(pos):
    if pd.isna(pos):
        return 'no_position_data'
    elif pos <= 3:
        return '1-3 (top)'
    elif pos <= 10:
        return '4-10'
    elif pos <= 20:
        return '11-20'
    else:
        return '21+'

def build_rule_df(raw_agg):
    d = raw_agg.copy()
    d['gsc_avg_position'] = d['gsc_sum_position'] / d['gsc_impressions']
    d.loc[d['gsc_sum_position'] == 0, 'gsc_avg_position'] = pd.NA

    signal = d[d['gsc_impressions'] > 0].copy()
    signal['ctr'] = signal['gsc_clicks'] / signal['gsc_impressions']
    signal['position_bucket'] = signal['gsc_avg_position'].apply(position_bucket)
    peer_ctr = signal.groupby('position_bucket')['ctr'].mean().to_dict()

    d['ctr'] = d['gsc_clicks'] / d['gsc_impressions']
    d['engagement_rate'] = d['ga4_engaged_sessions'] / d['ga4_sessions']
    d['position_bucket'] = d['gsc_avg_position'].apply(position_bucket)
    d['peer_avg_ctr'] = d['position_bucket'].map(peer_ctr)

    conditions = [
        (d['gsc_impressions'] >= 50) & d['ctr'].notna() & d['peer_avg_ctr'].notna()
            & (d['ctr'] < 0.5 * d['peer_avg_ctr']),
        (d['ga4_sessions'] >= 10) & d['engagement_rate'].notna()
            & (d['engagement_rate'] < 0.10),
    ]
    d['action'] = np.select(conditions, ['snippet_fix', 'content_fix'], default='monitor')
    d['reason_code'] = np.select(conditions,
        ['CTR_below_half_position_peers', 'engagement_below_10pct_reliable'], default='no_flag_triggered')
    d['rule_score'] = np.select(conditions,
        [(d['peer_avg_ctr'] - d['ctr']) * d['gsc_impressions'],
         d['ga4_sessions'] * (0.10 - d['engagement_rate']) * 10], default=0)

    eligible = (d['gsc_impressions'] >= 50) | (d['ga4_sessions'] >= 10)
    return d[eligible].copy()

march_rule_df = build_rule_df(df_march_agg)
feb_rule_df = build_rule_df(df_feb_agg)
print(f"March-eligible: {len(march_rule_df)}, Feb-eligible: {len(feb_rule_df)}")
print(march_rule_df['action'].value_counts())

March-eligible: 116512, Feb-eligible: 93871
action
snippet_fix    75663
monitor        28755
content_fix    12094
Name: count, dtype: int64


In [3]:
#decline label
MIN_CLICKS_FOR_LABEL = 5
DECLINE_THRESHOLD = 0.8

def build_labelable(rule_df, next_month_clicks_df, next_month_col):
    merged = rule_df.merge(next_month_clicks_df, on=['client_hash_id', 'content_hash_id'], how='left')
    tracked = merged[next_month_col].notna()
    lab = merged[tracked & (merged['gsc_clicks'] >= MIN_CLICKS_FOR_LABEL)].copy()
    lab['decline_label'] = (lab[next_month_col] < DECLINE_THRESHOLD * lab['gsc_clicks']).astype(int)
    return lab

labelable = build_labelable(march_rule_df, df_april_agg, 'gsc_clicks_apr')
print(f"March->April labelable: {len(labelable)}, base rate {labelable['decline_label'].mean():.3f}")

FEATURES = ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions',
            'ga4_engaged_sessions', 'ctr', 'engagement_rate', 'peer_avg_ctr']

def build_X(df, feature_cols):
    X = df[feature_cols].copy()
    worst = X['gsc_avg_position'].max()
    X['gsc_avg_position'] = X['gsc_avg_position'].fillna(worst + 10 if pd.notna(worst) else 100)
    X['ga4_sessions'] = X['ga4_sessions'].fillna(0)
    X['ga4_engaged_sessions'] = X['ga4_engaged_sessions'].fillna(0)
    X['engagement_rate'] = X['engagement_rate'].fillna(0)
    X['ctr'] = X['ctr'].fillna(0)
    X['peer_avg_ctr'] = X['peer_avg_ctr'].fillna(X['peer_avg_ctr'].median())
    X = X.join(pd.get_dummies(df['position_bucket'], prefix='pos', drop_first=True))
    return X

March->April labelable: 28805, base rate 0.545


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [5]:
#model
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

X_train = build_X(labelable, FEATURES)
y_train = labelable['decline_label'].values

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

deployed_model = LogisticRegression(max_iter=1000, random_state=0)
deployed_model.fit(X_train_scaled, y_train)

# score EVERY March-eligible page, not just the labelable subset
X_all = build_X(march_rule_df, FEATURES)
X_all = X_all.reindex(columns=X_train.columns, fill_value=0)
X_all_scaled = scaler.transform(X_all)
march_rule_df['logreg_prob'] = deployed_model.predict_proba(X_all_scaled)[:, 1]

print(f"Scored {len(march_rule_df)} eligible pages")
print(f"  ({len(labelable)} of those had a known April outcome and were used to fit the model)")
march_rule_df['logreg_prob'].describe()

Scored 116512 eligible pages
  (28805 of those had a known April outcome and were used to fit the model)


,logreg_prob
count,116512.000000
mean,0.541080
std,0.076669
min,0.000002
25%,0.523094
50%,0.565046
75%,0.585316
max,0.922834


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.